# Lab 1 — Regression Models

**DA380A Machine Learning · Group 28**

**Dataset:** `dataset3-AgriculturalProduction` — Indian crop statistics per state, district, crop, season and year (1997-98 to 2020-21).

**Target:** `Yield` — **tonnes harvested per hectare sown**. It measures how productive a field was, independent of its size. Predicting it tells a planner what harvest per hectare to expect for a given crop, season, place and year.

Why this dataset, and every cleaning decision, is documented in [`docs/step1-dataset-choice.md`](../docs/step1-dataset-choice.md).

| Section | Lab task |
|---|---|
| 1 | Explore and understand the dataset |
| 2 | Prepare the data |
| 3 | Create training and test data |
| 4 | Build at least two regression models |
| 5 | Evaluate the models |
| 6 | Improve and tune the models |
| 7 | Interpret and compare the results |

## 0. Setup

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import mean_absolute_error, r2_score, root_mean_squared_error
from sklearn.model_selection import GridSearchCV, cross_validate, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, StandardScaler

from src.overview import compare_datasets, load_dataset
from src.data import (CATEGORICAL, CSV_PATH, NUMERIC, RANDOM_STATE, TARGET,
                      load_agriculture, load_raw_tonnes)

SAMPLE_SIZE = 40_000   # lower to 20_000 for a faster run while experimenting
pd.set_option("display.max_columns", None)
sns.set_theme(style="whitegrid")

## 1. Explore and understand the dataset

### 1.1 Choosing the dataset
We compared the five provided datasets on: a clear numeric target, enough rows, useful features, data quality, how well we understand the domain, and leakage risk. The table below is the quick overview of every file (this cell reads all 11 CSV files and takes about a minute).

In [ ]:
overview = compare_datasets()
overview

`AgriculturalProduction` is a single file with a clear numeric target and a mix of categorical and numeric features, and it needs almost no text parsing. The other datasets either need several files joined first (VideoReviews, SalesTransactions1), have weak features (SalesTransactions2), or need every column parsed from text (AirTravel). Full scoring: `docs/step1-dataset-choice.md`.

### 1.2 First look at the raw data

In [ ]:
raw = load_dataset(CSV_PATH)
print(raw.shape)
raw.head()

In [ ]:
raw.info()

In [ ]:
raw.describe().T

In [ ]:
missing = (raw.isna().mean() * 100).round(2)
print("Missing values (%):")
print(missing[missing > 0])
print("\nFully duplicated rows:", raw.duplicated().sum())

In [ ]:
for col in ["Area Units", "Production Units", "Season"]:
    print(raw[col].value_counts(dropna=False), "\n")
print("Unique values:", raw[["State", "District", "Crop", "Year"]].nunique().to_dict())

### 1.3 Data-quality and leakage checks

Two things stand out in the raw data:

1. **Units:** `Area` is always in hectares, but `Production` is in **tonnes, bales or nuts**. A yield in bales per hectare can't be compared with tonnes per hectare, so we keep only the tonnes rows.
2. **`Yield` is calculated from two other columns.** If `Yield == Production / Area`, then keeping `Production` as a feature would let the model rebuild the formula instead of learning anything (**data leakage**). We check:

In [ ]:
tonnes = raw[raw["Production Units"] == "Tonnes"].dropna(subset=["Production", "Area", "Yield"])
difference = (tonnes["Yield"] - tonnes["Production"] / tonnes["Area"]).abs()
print(f"Largest |Yield - Production/Area|: {difference.max():.2e}")

The difference is ~0 (floating-point noise), so **`Yield` is exactly `Production / Area`**. `Production` is dropped from the features. Section 7.4 shows what happens if we don't.

### 1.4 Cleaning
All cleaning lives in `src/data.py` so it is applied the same way every time. Each step prints how many rows it removed:

In [ ]:
df, notes = load_agriculture(sample_size=SAMPLE_SIZE)
print("\n".join(notes))
print("\nCleaned dataset:", df.shape)
df.head()

In [ ]:
df.describe().T

### 1.5 The target: `Yield`

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
sns.histplot(df[TARGET], bins=80, ax=axes[0])
axes[0].set(title="Yield (t/ha)", xlabel="Yield")
sns.histplot(np.log1p(df[TARGET]), bins=80, ax=axes[1])
axes[1].set(title="log(1 + Yield)", xlabel="log1p(Yield)")
plt.tight_layout(); plt.show()

print(f"Skew: {df[TARGET].skew():.2f}   |   skew of log1p(Yield): {np.log1p(df[TARGET]).skew():.2f}")

### 1.6 Outliers

In [ ]:
q1, q3 = df[TARGET].quantile([0.25, 0.75])
upper_fence = q3 + 1.5 * (q3 - q1)
n_flagged = (df[TARGET] > upper_fence).sum()
print(f"IQR upper fence: {upper_fence:.2f} t/ha -> {n_flagged:,} rows ({n_flagged / len(df):.1%}) flagged as outliers")

top_crops = df["Crop"].value_counts().head(12).index
order = df[df["Crop"].isin(top_crops)].groupby("Crop")[TARGET].median().sort_values().index
plt.figure(figsize=(13, 5))
sns.boxplot(data=df[df["Crop"].isin(top_crops)], x="Crop", y=TARGET, order=order)
plt.yscale("log"); plt.xticks(rotation=45, ha="right")
plt.title("Yield per crop (12 most common crops, log scale)")
plt.tight_layout(); plt.show()

The IQR rule flags many rows, but the boxplot shows why: **each crop has its own yield level**. Sugarcane is harvested at tens of tonnes per hectare, pulses at around one tonne. These "outliers" are real, so we **keep** them and let the model learn them through the `Crop` feature. Only physically impossible values (above the 99.9th percentile, max 9,801 t/ha in the raw data) were removed in 1.4.

### 1.7 Relationships with the target

In [ ]:
print(df.groupby("Crop")[TARGET].median().sort_values(ascending=False).head(8).round(2))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 4.5))

season_order = df.groupby("Season")[TARGET].median().sort_values(ascending=False).index
sns.barplot(data=df, x="Season", y=TARGET, order=season_order, estimator="median", errorbar=None, ax=axes[0])
axes[0].set(title="Median yield per season", ylabel="Median Yield (t/ha)")
axes[0].tick_params(axis="x", rotation=30)

df.groupby("Year")[TARGET].median().plot(marker="o", ax=axes[1])
axes[1].set(title="Median yield per year", ylabel="Median Yield (t/ha)")

sample = df.sample(5_000, random_state=RANDOM_STATE)
sns.scatterplot(data=sample, x="Area", y=TARGET, alpha=0.25, s=10, ax=axes[2])
axes[2].set(xscale="log", yscale="log", title="Area vs Yield (5,000-row sample, log-log)")
plt.tight_layout(); plt.show()

print("Correlation with Yield (Pearson):", df[NUMERIC + [TARGET]].corr()[TARGET].drop(TARGET).round(3).to_dict())

In [ ]:
state_median = df.groupby("State")[TARGET].median().sort_values()
plt.figure(figsize=(13, 4))
state_median.plot(kind="bar")
plt.title("Median yield per state"); plt.ylabel("Median Yield (t/ha)")
plt.tight_layout(); plt.show()

### 1.8 Main observations

- After cleaning: **325,682 usable rows**; we work with a random sample of **40,000** to keep training time reasonable (5 columns + target).
- The raw data has **no duplicated rows** and very little missing data (1% of `Production`, 33 rows of `Area`/`Yield`/`Season`).
- **`Yield = Production / Area` exactly** → `Production` would leak the target and is removed.
- The target is **strongly right-skewed** (skew ≈ 5). Most yields are around 0.5–2.5 t/ha, while sugarcane, banana, tapioca, potato and onion reach 10–56 t/ha.
- **Crop is clearly the most important driver**: median yield ranges from about 0.1 t/ha (cardamom) to 56 t/ha (sugarcane).
- **Season matters**: "Whole Year" crops (median ≈ 3.7 t/ha) have far higher yields than Kharif (≈ 0.8) or Rabi (≈ 1.0) crops — partly because whole-year crops include sugarcane and banana.
- **Year** shows a slow upward trend (median ≈ 0.9 t/ha in 1997 → ≈ 1.2 in 2020), but its linear correlation with yield is tiny (≈ 0.04).
- **Area** is extremely skewed (median ≈ 570 ha, top 1% above 180,000 ha) and has almost no *linear* relationship with yield (r ≈ −0.02). We use `log(1 + Area)` so the size differences don't dominate.
- **State** shows clear differences in median yield, so location carries information.

## 2. Prepare the data

**Features (X):** `State`, `Crop`, `Season` (categorical) and `Year`, `Area` (numeric). **Target (y):** `Yield`.

Already done in `load_agriculture()` (section 1.4), because these steps don't learn anything from the data:
- kept only rows measured in tonnes · dropped missing targets, `Yield == 0` and impossible yields · turned `"1997-98"` into `1997`
- dropped `Production` (leakage), `Area Units` / `Production Units` (constant), `District` (728 categories; `State` keeps the geographic information)

Steps that **learn from the data** (medians, means, category lists) go into a `Pipeline`, so they are fitted on the training set only — inside every cross-validation fold too. That prevents information from the test set leaking into training.

| Column type | Step | Why |
|---|---|---|
| Numeric | `SimpleImputer(median)` | No missing values right now, but new data might have some; the median isn't pulled by outliers |
| Numeric | `log1p` | `Area` spans 0.004 to 8.5 million hectares; the log stops the largest farms from dominating |
| Numeric | `StandardScaler` | Linear models (Ridge) are sensitive to scale; the random forest doesn't need it but isn't hurt by it |
| Categorical | `SimpleImputer(most_frequent)` | Safety net for missing categories |
| Categorical | `OneHotEncoder(handle_unknown="ignore")` | State, crop and season have no natural order, so ordinal numbers would invent one; unseen categories in new data won't crash the model |

In [ ]:
X = df.drop(columns=TARGET)
y = df[TARGET]
print("X:", X.shape, "| y:", y.shape)
X.dtypes

In [ ]:
numeric_pipeline = Pipeline([
    ("impute", SimpleImputer(strategy="median")),
    ("log", FunctionTransformer(np.log1p, feature_names_out="one-to-one")),
    ("scale", StandardScaler()),
])
categorical_pipeline = Pipeline([
    ("impute", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
])
preprocess = ColumnTransformer([
    ("num", numeric_pipeline, NUMERIC),
    ("cat", categorical_pipeline, CATEGORICAL),
])
preprocess

## 3. Create training and test data

80% training, 20% test, split **before** anything is fitted. The test set is not used again until section 5. Model comparison and tuning use **5-fold cross-validation on the training set**, so no separate validation set is needed.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE)
print("Train:", X_train.shape, "| Test:", X_test.shape)
print(f"Mean yield — train: {y_train.mean():.2f}, test: {y_test.mean():.2f}")

In [ ]:
# Check the preprocessing on the training set only
X_train_prepared = preprocess.fit_transform(X_train)
print("Features after preprocessing:", X_train_prepared.shape[1])
print("Missing values after preprocessing:", np.isnan(X_train_prepared).sum())
preprocess.get_feature_names_out()[:10]

## 4. Build the regression models

Three models, all trained on the same training data, each wrapped in a pipeline with the same preprocessing:

| Model | Why |
|---|---|
| **Dummy (mean)** | Always predicts the mean yield. Any useful model must beat it |
| **Linear Regression** | Simple, fast and interpretable: one coefficient per feature. Assumes effects add up linearly |
| **Random Forest** | Many decision trees averaged together. Captures non-linear effects and interactions (e.g. a crop that yields more in one season) |

They are compared with **5-fold cross-validation on the training set** (scikit-learn reports errors as negative numbers, so we flip the sign). The forest uses `n_jobs=1` and the cross-validation runs the 5 folds in parallel instead — this takes a few minutes.

In [ ]:
def build_pipeline(model):
    """Fresh copy of the preprocessing + a fresh copy of the model, so pipelines never share fitted state."""
    return Pipeline([("prep", clone(preprocess)), ("model", clone(model))])


models = {
    "Dummy (mean)": DummyRegressor(strategy="mean"),
    "Linear Regression": LinearRegression(),
    "Random Forest": RandomForestRegressor(n_estimators=100, random_state=RANDOM_STATE, n_jobs=1),
}
scoring = {"MAE": "neg_mean_absolute_error", "RMSE": "neg_root_mean_squared_error", "R2": "r2"}

cv_rows = []
for name, model in models.items():
    cv = cross_validate(build_pipeline(model), X_train, y_train, cv=5, scoring=scoring, n_jobs=-1)
    cv_rows.append({
        "Model": name,
        "CV MAE": -cv["test_MAE"].mean(),
        "CV RMSE": -cv["test_RMSE"].mean(),
        "CV R2": cv["test_R2"].mean(),
        "CV R2 std": cv["test_R2"].std(),
    })

cv_results = pd.DataFrame(cv_rows).set_index("Model").round(3)
cv_results

## 5. Evaluate the models on the test set

Each model is fitted on the full training set and evaluated **once** on the same test set.

- **MAE** — average size of the error, in t/ha. Easy to explain.
- **RMSE** — like MAE but squares errors first, so big misses count much more. Also in t/ha.
- **R²** — share of the variation in yield the model explains (1 = perfect, 0 = no better than predicting the mean).

In [ ]:
def regression_metrics(y_true, y_pred):
    return {
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": root_mean_squared_error(y_true, y_pred),
        "R2": r2_score(y_true, y_pred),
    }


fitted = {}
test_rows = []
for name, model in models.items():
    pipe = build_pipeline(model)
    if name == "Random Forest":
        pipe.set_params(model__n_jobs=-1)   # one model at a time here, so let it use all cores
    pipe.fit(X_train, y_train)
    fitted[name] = pipe
    test_rows.append({"Model": name, **regression_metrics(y_test, pipe.predict(X_test))})

test_results = pd.DataFrame(test_rows).set_index("Model").round(3)
test_results

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, name in zip(axes, ["Linear Regression", "Random Forest"]):
    pred = fitted[name].predict(X_test)
    ax.scatter(y_test, pred, alpha=0.2, s=8)
    lims = [min(0, pred.min()), y_test.max()]
    ax.plot(lims, lims, "r--", label="perfect prediction")
    # same symmetric-log scale on both axes: readable for 0.1-100 t/ha and still shows negative predictions
    ax.set_xscale("symlog", linthresh=0.1)
    ax.set_yscale("symlog", linthresh=0.1)
    ax.set(xlabel="Actual yield (t/ha)", ylabel="Predicted yield (t/ha)",
           title=f"{name}  (R² = {r2_score(y_test, pred):.2f})")
    ax.legend()
plt.tight_layout(); plt.show()

_Interpretation (fill in from the numbers above):_ the Dummy model has R² ≈ 0 by definition. Linear Regression explains roughly three quarters of the variation; the Random Forest explains over 90% and roughly halves the typical error. The scatter plots show the linear model struggling at the extremes (it can even predict negative yields), while the forest's predictions follow the diagonal much more closely.

## 6. Improve and tune the models

`GridSearchCV` tries every combination of the listed hyperparameters with **5-fold cross-validation on the training set**, and keeps the one with the lowest RMSE. The test set is only used afterwards to compare *before vs after*.

- **Linear model → Ridge.** Plain Linear Regression has no hyperparameters, so we tune its regularised version: `alpha` controls how strongly the coefficients are shrunk (α = 0 is plain linear regression).
- **Random Forest.** `max_features` = share of features considered at each split (lower = more diverse trees, faster); `min_samples_leaf` = minimum rows per leaf (higher = smoother, less overfitting).

In [ ]:
ridge_search = GridSearchCV(
    build_pipeline(Ridge()),
    {"model__alpha": [0.01, 0.1, 1, 10, 100]},
    cv=5, scoring="neg_root_mean_squared_error", n_jobs=-1)
ridge_search.fit(X_train, y_train)
print("Best Ridge:", ridge_search.best_params_, f"| CV RMSE {-ridge_search.best_score_:.3f}")

In [ ]:
rf_search = GridSearchCV(
    build_pipeline(RandomForestRegressor(n_estimators=100, random_state=RANDOM_STATE, n_jobs=1)),
    {
        "model__max_features": [0.3, 0.6, 1.0],
        "model__min_samples_leaf": [1, 4],
    },
    cv=5, scoring="neg_root_mean_squared_error", n_jobs=-1)
rf_search.fit(X_train, y_train)
print("Best Random Forest:", rf_search.best_params_, f"| CV RMSE {-rf_search.best_score_:.3f}")

pd.DataFrame(rf_search.cv_results_)[["params", "mean_test_score", "mean_fit_time"]] \
    .assign(CV_RMSE=lambda d: -d["mean_test_score"]).drop(columns="mean_test_score") \
    .sort_values("CV_RMSE").round(3)

In [ ]:
tuned = {
    "Linear Regression": ("Ridge (tuned)", ridge_search.best_estimator_),
    "Random Forest": ("Random Forest (tuned)", rf_search.best_estimator_),
}
rows = []
for before_name, (after_name, after_model) in tuned.items():
    rows.append({"Model": before_name, "Version": "before",
                 "CV RMSE": cv_results.loc[before_name, "CV RMSE"],
                 **test_results.loc[before_name].to_dict()})
    search = ridge_search if before_name == "Linear Regression" else rf_search
    rows.append({"Model": after_name, "Version": "after",
                 "CV RMSE": -search.best_score_,
                 **regression_metrics(y_test, after_model.predict(X_test))})

before_after = pd.DataFrame(rows).set_index("Model").round(3)
before_after

_Discussion (fill in from the table):_ say whether tuning lowered the CV RMSE and the test RMSE, and by how much. It is fine — and honest — if the gain is small: the default random forest is already a strong model, and Ridge can only shrink coefficients, not fix the linear model's main limitation (it can't capture crop × season interactions or the skewed target).

## 7. Interpret and compare the results

### 7.1 Most influential features — permutation importance
For each original column we **shuffle its values in the test set** and measure how much the RMSE gets worse. A big increase means the model relies heavily on that column. This works for any model and on the original columns (not the one-hot pieces), so the two models can be compared directly.

In [ ]:
final_models = {"Ridge (tuned)": ridge_search.best_estimator_, "Random Forest (tuned)": rf_search.best_estimator_}

importance = {}
for name, model in final_models.items():
    result = permutation_importance(model, X_test, y_test, scoring="neg_root_mean_squared_error",
                                    n_repeats=5, random_state=RANDOM_STATE, n_jobs=-1)
    importance[name] = pd.Series(result.importances_mean, index=X_test.columns)

importance = pd.DataFrame(importance).sort_values("Random Forest (tuned)", ascending=False)
importance.plot(kind="barh", figsize=(9, 4))
plt.gca().invert_yaxis()
plt.xlabel("Increase in RMSE when the column is shuffled (t/ha)")
plt.title("Permutation importance on the test set")
plt.tight_layout(); plt.show()
importance.round(3)

### 7.2 Ridge coefficients (the linear model's own view)

In [ ]:
ridge = ridge_search.best_estimator_
coefs = pd.Series(ridge.named_steps["model"].coef_,
                  index=ridge.named_steps["prep"].get_feature_names_out())
print("Largest positive effects:\n", coefs.sort_values(ascending=False).head(8).round(2))
print("\nLargest negative effects:\n", coefs.sort_values().head(5).round(2))

### 7.3 Residuals — where do the models go wrong?

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5), sharey=True)
for ax, (name, model) in zip(axes, final_models.items()):
    pred = model.predict(X_test)
    ax.scatter(pred, y_test - pred, alpha=0.2, s=8)
    ax.axhline(0, color="red", linestyle="--")
    ax.set(xscale="symlog", xlabel="Predicted yield (t/ha)", ylabel="Actual − predicted", title=name)
plt.tight_layout(); plt.show()

errors = X_test.assign(abs_error=(y_test - rf_search.best_estimator_.predict(X_test)).abs())
print("Random Forest — mean absolute error per crop (worst 6):")
print(errors.groupby("Crop")["abs_error"].mean().sort_values(ascending=False).head(6).round(2))

### 7.4 Why `Production` had to go (leakage demonstration)
Because `Yield = Production / Area`, taking logs gives **log(Yield) = log(Production) − log(Area)** — a perfectly linear relationship. A plain linear regression on those two columns should therefore find it exactly:

In [ ]:
leaky = load_raw_tonnes().dropna(subset=["Production", "Area", "Yield"])
leaky = leaky[(leaky["Production"] > 0) & (leaky["Area"] > 0)].sample(SAMPLE_SIZE, random_state=RANDOM_STATE)

X_leak = np.log(leaky[["Production", "Area"]])
y_leak = np.log(leaky[TARGET])
Xl_train, Xl_test, yl_train, yl_test = train_test_split(
    X_leak, y_leak, test_size=0.2, random_state=RANDOM_STATE)

leak_model = LinearRegression().fit(Xl_train, yl_train)
print(f"R² on the test set: {r2_score(yl_test, leak_model.predict(Xl_test)):.4f}")
print("Coefficients:", dict(zip(X_leak.columns, leak_model.coef_.round(4))),
      "| intercept:", round(leak_model.intercept_, 4))

A **perfect score** (R² = 1.0000) with coefficients **+1 for Production and −1 for Area** and intercept 0: the model has simply rediscovered the formula. That score is worthless in practice — when you want to *predict* a yield, the harvest (`Production`) isn't known yet. This is why `Production` is excluded from every model above.

### 7.5 Strengths and weaknesses

| | Linear / Ridge Regression | Random Forest |
|---|---|---|
| **Accuracy here** | Lower — misses non-linear effects and crop × season interactions | Much higher |
| **Interpretability** | High — one coefficient per feature | Lower — needs permutation importance to explain |
| **Training speed** | Under a second | Minutes with cross-validation |
| **Extrapolation** | Can predict impossible values (negative yields) | Predictions stay within the range seen in training |
| **Skewed target** | Struggles; errors dominated by high-yield crops | Copes well — each leaf predicts a local average |
| **Preprocessing needs** | Needs scaling and one-hot encoding | Works without scaling |

### 7.6 Recommendation

_Fill in with your final numbers:_ we recommend the **tuned Random Forest**. On the untouched test set it achieves MAE ≈ … t/ha, RMSE ≈ … t/ha and R² ≈ …, against R² ≈ … for Ridge. The most influential features are **Crop** first, then … . Its weakness is lower interpretability, which permutation importance partly compensates for.

**Possible improvements:** add `District` (with grouping of rare districts), predict `log(Yield)` so high-yield crops don't dominate the error, use the full 325k rows, or try gradient boosting.